# 01_merge.ipynb
**Purpose:** Clean the raw selected data — enforce numeric types, drop rows missing the target, encode categoricals — and write a model-ready CSV.

| | |
|---|---|
| **Input** | `data/raw_selected.csv` — output of `00_pull.ipynb` |
| **Output** | `data/analysis_clean.csv` — fully numeric, no missing target |
| **Key functions** | `load_raw()`, `clean_types()`, `encode_categoricals()`, `drop_missing_target()` |

In [1]:
# Imports 
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
#Paths
ROOT       = Path.cwd().parent
DATA_DIR   = ROOT / "data"
OUTPUT_DIR = ROOT / "output"

INPUT_FILE  = DATA_DIR / "raw_selected.csv" 
OUTPUT_FILE = DATA_DIR / "analysis_clean.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'hourly_wage'

In [3]:
# Functions 
def load_raw(path: Path) -> pd.DataFrame:
    """Load the raw selected CSV produced by 00_pull."""
    df = pd.read_csv(path)
    print(f"[load_raw] Shape: {df.shape}")
    print(f"[load_raw] Missing per column:\n{df.isna().sum()}")
    return df


def clean_types(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce all columns to numeric where possible; non-parseable → NaN."""
    df = df.copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"[clean_types] dtypes after coercion:\n{df.dtypes}")
    return df


def encode_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """Convert remaining pandas Categorical columns to integer codes."""
    df = df.copy()
    cat_cols = df.select_dtypes(include='category').columns.tolist()
    for col in cat_cols:
        df[col] = df[col].cat.codes
    print(f"[encode_categoricals] Encoded: {cat_cols}")
    return df


def drop_missing_target(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """Drop rows where the target variable is NaN."""
    before = len(df)
    df = df.dropna(subset=[target])
    print(f"[drop_missing_target] Dropped {before - len(df)} rows → {len(df)} remain")
    return df

In [4]:
# Diagnostic before merge/clean 
df_raw = load_raw(INPUT_FILE)
print("\n BEFORE CLEAN")
print(df_raw.shape)
print(df_raw.head())

[load_raw] Shape: (5192, 22)
[load_raw] Missing per column:
hourly_wage                  0
education                    0
gender                       0
hours_cat                    0
experience                   0
depression_score             0
hh_income                    0
year_born                    0
region                       0
full_time_status             0
job_role                     0
months_per_year              0
race                         0
hispanic_latino              0
country_of_birth             0
works_public_school          0
urban_density                0
college_for_pd               0
workshop_for_pd              0
enrolled_in_college          0
interaction_quality_total    0
self_rated_health            0
dtype: int64

 BEFORE CLEAN
(5192, 22)
                                       hourly_wage  \
0  Not enough information to calculate hourly wage   
1                                    8.65384615385   
2                                    4.44444444445   
3  

In [5]:
summary_vars = [
    "hourly_wage",
    "depression_score",
    "experience",
    "hh_income",
    "interaction_quality_total",
]
df_raw[summary_vars].describe().round(2)

,hourly_wage,depression_score,experience,hh_income,interaction_quality_total
count,5192,5192,5192,5192,5192
unique,1277,40,8,7,7
top,WF spawned from center with only administrativ...,0.0,<=5 years,"$15,001 to $30,000",All items correct
freq,483,1724,1279,1209,2373


In [6]:
print(df_raw.shape)
print(df_raw.head())

df_raw.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved to: {OUTPUT_FILE}")

(5192, 22)
                                       hourly_wage  \
0  Not enough information to calculate hourly wage   
1                                    8.65384615385   
2                                    4.44444444445   
3                                             10.0   
4                                              8.5   

                           education  gender         hours_cat  \
0          Associate degree (AA, AS)  Female     Over 40 hours   
1          Associate degree (AA, AS)  Female          40 hours   
2     GED or high school equivalency  Female     Over 40 hours   
3          Associate degree (AA, AS)  Female     Over 40 hours   
4  Some college credit but no degree  Female  20 hours or less   

          experience depression_score                     hh_income  \
0  >20 to <=25 years              4.0  Don't Know/Refused/No Answer   
1          >25 years              0.0            $15,001 to $30,000   
2  >10 to <=15 years             17.0            $15,0